# code for topic modeling of episode annotations and recall transcripts

### imports

In [1]:
import numpy as np
import pandas as pd
import hypertools as hyp
import os
import re
import pickle
from num2words import num2words
from scipy.signal import resample
from scipy.interpolate import interp1d

### paths

In [2]:
data_dir = '../../data'
annot_dir = f'{data_dir}/annotations_dfs/'
transc_dir = f'{data_dir}/transcriptions/automatic/'
ep_traj_dir = f'{data_dir}/models/episodes/trajectories/'
rec_traj_dir = f'{data_dir}/models/participants/trajectories/'
pickle_dir = f'{data_dir}/pickles/'

### load formatted annotations

In [3]:
atlep1_df = pd.read_csv(annot_dir+'atlep1.csv')
atlep2_df = pd.read_csv(annot_dir+'atlep2.csv')
arrdev_df = pd.read_csv(annot_dir+'arrdev.csv')

### topic modeling parameters

In [4]:
n_topics = 100
episode_wsize = 50
recall_wsize = 200

# vectorizer parameters
vectorizer_params = {
    'model' : 'CountVectorizer', 
    'params' : {
        'stop_words' : 'english'
    }
}

# topic model parameters
semantic_params = {
    'model' : 'LatentDirichletAllocation', 
    'params' : {
        'n_components' : n_topics,
        'learning_method' : 'batch',
        'random_state' : 0,
    }
}

## functions

### for creating episode/recall sliding windows

In [5]:
def format_episode_text(textlist):
    """
    standardize annotation text format for modeling
    """
    formatted = []
    for chunk in textlist:
        lower_nopunc = re.sub("[^\w\s-]+", '', chunk.lower())    # remove all punctuation except for dashes
        no_acc = lower_nopunc.replace('É', 'E')    # remove accented characters
        no_digit = re.sub(r"(\d+)", lambda x: num2words(int(x.group(0))), no_acc)    # convert digits to words
        spaced = ' '.join(no_digit.replace(',', ' ').split())    # deal with inconsistent whitespace
        formatted.append(spaced)
    
    return formatted

In [6]:
def get_episode_windows(episode_df, episode_wsize=episode_wsize):
    # throw all annotations into "bag of words"
    episode_bag = format_episode_text(episode_df.loc[:,'Narrative details (external events)':'Setting'].apply(
        lambda x: ', '.join(x.fillna('')), axis=1).values.tolist())
    # create sliding windows
    episode_w = []
    for idx, sentence in enumerate(episode_bag):
        episode_w.append(' '.join(episode_bag[idx:idx+episode_wsize]))

    return episode_w

In [7]:
def get_recall_windows(transcript, recall_wsize=recall_wsize):
    rec_list = transcript.split()
    # create sliding windows
    recall_w = []
    for ix, word in enumerate(rec_list):
        recall_w.append(' '.join(rec_list[ix:ix+recall_wsize]))
        
    return recall_w

### for interpolating episode trajectories

In [8]:
def find_midpoint_time(df):
    """
    returns list of timepoints at middle of each annotation segment
    """
    # use timestamp of last frame from episode to find midpoint of last annotation
    df_shapes = [atlep1_df.shape, atlep2_df.shape, arrdev_df.shape]
    endframe_times = [1466.0, 1316.52, 1236.6]
    endframe_time = endframe_times[df_shapes.index(df.shape)]

    midpoint_times = []
    for i, tpt in enumerate(df['Onset time']):
        try:
            midpoint_times.append((tpt + df['Onset time'][i+1]) / 2)
        except KeyError:    # handle last annotation
            midpoint_times.append((tpt + endframe_time) / 2)
                    
    return midpoint_times, endframe_time

In [9]:
def interpolate_episode(traj, df, resolution=1):
    """
    uses linear interpolation to resample episode trajectory timeseries to desired resolution. 
    'resolution' is in units of seconds.
    """
    # get middle timepoint for each annotation
    midpoint_times, endframe_time = find_midpoint_time(df)
    new_traj = np.arange(int(round(endframe_time)), step=resolution)
    interp_func = interp1d(midpoint_times, traj, axis=0, fill_value='extrapolate')
    
    return interp_func(new_traj)

## main topic modeling function

In [10]:
def fit_and_transform(documents, vec_params=vectorizer_params, sem_params=semantic_params, 
                        corpus=None, resample_shape=None, return_windows=False):
    # handle annotations
    if isinstance(documents, pd.DataFrame):
        windows = get_episode_windows(documents, episode_wsize)
        corpus = windows if not corpus else corpus
        # fit topic model and transform documents
        traj =  hyp.tools.format_data(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
        
        # interpolate to length of episode (seconds)
        if return_windows:
            return interpolate_episode(traj, documents), windows
        else:
            return interpolate_episode(traj, documents)
        
    # handle recall transcripts
    elif isinstance(documents, str):
        if not corpus:
            raise ValueError("You must pass a training corpus to transform recall transcripts")
        windows = get_recall_windows(documents, recall_wsize)
        traj =  hyp.tools.format_data(windows, vectorizer=vec_params, semantic=sem_params, corpus=corpus)[0]
        
        # resample to corresponding episode length
        return resample(traj, resample_shape) 

## model episode content, get sliding windows for fitting recall models

In [11]:
atlep1_traj, atlep1_windows = fit_and_transform(atlep1_df, return_windows=True)
atlep2_traj, atlep2_windows = fit_and_transform(atlep2_df, return_windows=True)
arrdev_traj, arrdev_windows = fit_and_transform(arrdev_df, return_windows=True)

## save episode trajectories/load saved trajectories

In [12]:
# np.save(os.path.join(ep_traj_dir, 'atlep1_trajectory'), atlep1_traj)
# np.save(os.path.join(ep_traj_dir, 'atlep2_trajectory'), atlep2_traj)
# np.save(os.path.join(ep_traj_dir, 'arrdev_trajectory'), arrdev_traj)

atlep1_traj = np.load(os.path.join(ep_traj_dir, 'atlep1_trajectory.npy'))
atlep2_traj = np.load(os.path.join(ep_traj_dir, 'atlep2_trajectory.npy'))
arrdev_traj = np.load(os.path.join(ep_traj_dir, 'arrdev_trajectory.npy'))

## load in and model recall transcripts

In [13]:
participant_trajectories = {
    'atlep1' : [],
    'prediction' : [],
    'delayed' : [],
    'atlep2' : [],
    'arrdev' : []
}

total = sum([len([f for f in files if f.endswith('corrected.wav.txt')]) for r, d, files in os.walk(transc_dir)])
# walk transcription directory structure
currfile = 1
for root, dirs, files in os.walk(transc_dir):
    
    # FOR USE WITH AUTOMATIC TRANSCRIPTS -- REMOVE WHEN SWITCHING TO MANUAL
    transcripts = [f for f in files if f.endswith('corrected.wav.txt')]
    for transc in transcripts:

        # assign correct episode windows, corresponding episode trajectory shape, and dict key
        if any('prediction' in t for t in transcripts) or 'delayed' in transc:
            corpus = atlep1_windows
            resample_shape = atlep1_traj.shape[0]
            if 'recall' in transc:
                rectype = 'atlep1'
            elif 'prediction' in transc:
                rectype = 'prediction'
            elif 'delayed' in transc:
                rectype = 'delayed'
            else:
                raise ValueError('Transcript is not a recognized option')
            
        elif '-A-' in root:
            corpus = atlep2_windows
            resample_shape = atlep2_traj.shape[0]
            rectype = 'atlep2'
            
        else:
            corpus = arrdev_windows
            resample_shape = arrdev_traj.shape[0]
            rectype = 'arrdev'
            
        with open(os.path.join(root,transc), 'r') as f:
            # FOR USE WITH AUTOMATIC TRANSCRIPTS -- REMOVE WHEN SWITCHING TO MANUAL
            transcript = ' '.join([line.split(',')[0].lower() for line in f.read().split('\n')])
            
        # fit topic model to episode annotations, 
        print(f'modeling transcript {currfile}/{total}...    {transc}')
        p_traj = fit_and_transform(transcript, resample_shape=resample_shape, corpus=corpus)
        
        participant_trajectories[rectype].append((transc.split('-')[0],p_traj))
        currfile += 1

modeling transcript 1/232...    debugF2QKD:debug9rZa3-recall-corrected.wav.txt
modeling transcript 2/232...    debugF2QKD:debug9rZa3-delayed-corrected.wav.txt
modeling transcript 3/232...    debugQ0oPh:debugCquQc-prediction-corrected.wav.txt
modeling transcript 4/232...    debugQ0oPh:debugCquQc-recall-corrected.wav.txt
modeling transcript 5/232...    debugmFRWe:debug4RZkW-prediction-corrected.wav.txt
modeling transcript 6/232...    debugmFRWe:debug4RZkW-recall-corrected.wav.txt
modeling transcript 7/232...    debugBhGxH:debugMWNLW-delayed-corrected.wav.txt
modeling transcript 8/232...    debugBhGxH:debugMWNLW-recall-corrected.wav.txt
modeling transcript 9/232...    debug92cgv:debugvdAIT-recall-corrected.wav.txt
modeling transcript 10/232...    debug92cgv:debugvdAIT-prediction-corrected.wav.txt
modeling transcript 11/232...    debugIFCgX:debugt0bgV-delayed-corrected.wav.txt
modeling transcript 12/232...    debugIFCgX:debugt0bgV-recall-corrected.wav.txt
modeling transcript 13/232...    d

modeling transcript 102/232...    debugTYPE4:debugnAJ5U-recall-corrected.wav.txt
modeling transcript 103/232...    debugoVPgV:debugSiRLD-recall-corrected.wav.txt
modeling transcript 104/232...    debugoVPgV:debugSiRLD-prediction-corrected.wav.txt
modeling transcript 105/232...    debug6GN7B:debugBfGog-delayed-corrected.wav.txt
modeling transcript 106/232...    debug6GN7B:debugBfGog-recall-corrected.wav.txt
modeling transcript 107/232...    debugHKLdw:debugk43rK-prediction-corrected.wav.txt
modeling transcript 108/232...    debugHKLdw:debugk43rK-recall-corrected.wav.txt
modeling transcript 109/232...    debugpdN5k:debug3flgN-recall-corrected.wav.txt
modeling transcript 110/232...    debugpdN5k:debug3flgN-prediction-corrected.wav.txt
modeling transcript 111/232...    debug5vYxL:debug5nmdy-recall-corrected.wav.txt
modeling transcript 112/232...    debug5vYxL:debug5nmdy-delayed-corrected.wav.txt
modeling transcript 113/232...    debugm7zD9:debugOLIFw-delayed-corrected.wav.txt
modeling tran

modeling transcript 202/232...    debug3dOrm:debugAjPUS-recall-corrected.wav.txt
modeling transcript 203/232...    debugdHg9G:debugnb698-recall-corrected.wav.txt
modeling transcript 204/232...    debugdHg9G:debugnb698-delayed-corrected.wav.txt
modeling transcript 205/232...    debugVFMcA:debugjM9rk-recall-corrected.wav.txt
modeling transcript 206/232...    debugVFMcA:debugjM9rk-delayed-corrected.wav.txt
modeling transcript 207/232...    debug3lixg:debugdwpeu-recall-corrected.wav.txt
modeling transcript 208/232...    debug3lixg:debugdwpeu-prediction-corrected.wav.txt
modeling transcript 209/232...    debugGaDml:debugFTHoY-recall-corrected.wav.txt
modeling transcript 210/232...    debugGaDml:debugFTHoY-prediction-corrected.wav.txt
modeling transcript 211/232...    debugBB0yq:debug30NBt-delayed-corrected.wav.txt
modeling transcript 212/232...    debugBB0yq:debug30NBt-recall-corrected.wav.txt
modeling transcript 213/232...    debug5crvm:debug703dt-delayed-corrected.wav.txt
modeling transcr

## save individual trajectories

In [14]:
for rectype, data in participant_trajectories.items():
    for (turkid, traj) in data:
        np.save(os.path.join(rec_traj_dir, rectype, f'{turkid}.npy'), traj)

## create and save average participant trajectories

In [35]:
for rectype, data in participant_trajectories.items():
    avg_trajectory = np.array([traj for (turkid, traj) in data]).mean(axis=0)
    np.save(os.path.join(rec_traj_dir, rectype, 'avg_trajectory.npy'), avg_trajectory)